In [0]:
%pip install openai sentence-transformers "chromadb==0.5.23" "numpy<2.0"
dbutils.library.restartPython()

In [0]:
import ast
import os
import hashlib
import shutil
import textwrap
from pathlib import Path
from typing import List, Dict
from sentence_transformers import SentenceTransformer
import chromadb

print("Imports OK")

In [0]:
CHROMA_PATH = "/tmp/portfolio_assistant/chroma_store"
BACKUP_ZIP  = "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/chroma_backup.zip"

db_file = os.path.join(CHROMA_PATH, "chroma.sqlite3")
if not os.path.exists(db_file):
    if os.path.exists(BACKUP_ZIP):
        os.makedirs(CHROMA_PATH, exist_ok=True)
        shutil.unpack_archive(BACKUP_ZIP, CHROMA_PATH)
        print("✓ Restored from backup")

model         = SentenceTransformer("BAAI/bge-small-en-v1.5")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Separate collection for code — keeps it isolated from prose retrieval
code_collection = chroma_client.get_or_create_collection(
    name="code_base",
    metadata={"hnsw:space": "cosine"}
)

print(f"knowledge_base: {chroma_client.get_or_create_collection('knowledge_base').count()} chunks")
print(f"code_base:      {code_collection.count()} chunks")

In [0]:
def chunk_python_file(filepath: Path) -> List[Dict]:
    """
    Parse a Python file and extract each function and class as a separate chunk.

    Why AST and not regex?
    - AST understands Python syntax — it won't get confused by functions
      inside strings, comments, or nested definitions
    - Gives us the exact line numbers so we can extract clean source code
    - Handles edge cases (decorators, multiline signatures) correctly

    Each chunk includes:
    - The full source code of the function/class
    - Metadata: file path, name, type (function/class), line number
    """
    try:
        source = filepath.read_text(encoding="utf-8")
        tree   = ast.parse(source)
    except (SyntaxError, UnicodeDecodeError) as e:
        print(f"  Skipping {filepath.name}: {e}")
        return []

    lines  = source.splitlines()
    chunks = []

    for node in ast.walk(tree):
        # Only extract top-level functions and classes
        # Skip nested functions — they'll be included in their parent class/function
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            continue

        # Only process top-level definitions (no parent function/class)
        # ast.walk gives all nodes — we filter to top-level only
        start_line = node.lineno - 1        # ast is 1-indexed, lists are 0-indexed
        end_line   = node.end_lineno        # exclusive

        # Include decorators (they're part of the function definition)
        if node.decorator_list:
            start_line = node.decorator_list[0].lineno - 1

        code = "\n".join(lines[start_line:end_line])

        # Skip tiny stubs — less than 3 lines aren't worth indexing
        if len(code.strip().splitlines()) < 3:
            continue

        node_type = "class" if isinstance(node, ast.ClassDef) else "function"

        # Stable ID: hash of filepath + function name
        chunk_id = hashlib.md5(
            f"{filepath}::{node.name}".encode()
        ).hexdigest()

        # The text we embed = file context + code
        # Adding the file path helps retrieval — "data_fetcher get_latest_signals"
        # is more meaningful than just the raw code
        embed_text = (
            f"File: {filepath.name}\n"
            f"{node_type.capitalize()}: {node.name}\n\n"
            f"{code}"
        )

        chunks.append({
            "id":        chunk_id,
            "text":      embed_text,
            "code":      code,           # raw code — shown to user
            "source":    str(filepath),
            "filename":  filepath.name,
            "name":      node.name,
            "type":      node_type,
            "line":      node.lineno
        })

    return chunks


def chunk_js_file(filepath: Path) -> List[Dict]:
    """
    Extract JavaScript functions using regex.
    JS doesn't have a clean built-in AST parser in Python,
    so we use regex for common function patterns.
    Covers: function declarations, arrow functions, method definitions.
    """
    source = filepath.read_text(encoding="utf-8")
    chunks = []

    # Match: function name(...) { ... }
    # and:   const name = (...) => { ... }
    # and:   name(...) { ... }  (class methods)
    pattern = re.compile(
        r'(?:^|\n)'                          # start of line
        r'(?:export\s+)?'                    # optional export
        r'(?:async\s+)?'                     # optional async
        r'(?:function\s+(\w+)|'             # function declaration
        r'(?:const|let|var)\s+(\w+)\s*=\s*(?:async\s+)?(?:\([^)]*\)|\w+)\s*=>|'  # arrow
        r'(\w+)\s*\([^)]*\)\s*\{)',         # method
        re.MULTILINE
    )

    for match in pattern.finditer(source):
        name = match.group(1) or match.group(2) or match.group(3)
        if not name or name in {"if", "for", "while", "switch"}:
            continue

        start = match.start()
        # Find the function body by counting braces
        brace_count = 0
        end = start
        for i, char in enumerate(source[start:], start):
            if char == "{":
                brace_count += 1
            elif char == "}":
                brace_count -= 1
                if brace_count == 0:
                    end = i + 1
                    break

        code = source[start:end].strip()
        if len(code.splitlines()) < 3:
            continue

        chunk_id  = hashlib.md5(f"{filepath}::{name}".encode()).hexdigest()
        embed_text = f"File: {filepath.name}\nFunction: {name}\n\n{code}"

        chunks.append({
            "id":       chunk_id,
            "text":     embed_text,
            "code":     code,
            "source":   str(filepath),
            "filename": filepath.name,
            "name":     name,
            "type":     "function",
            "line":     source[:start].count("\n") + 1
        })

    return chunks

In [0]:
import re  # needed for JS chunking

# File types to index and their chunkers
CHUNKERS = {
    ".py": chunk_python_file,
    ".js": chunk_js_file,
}

# Files/dirs to skip — not useful for retrieval
SKIP_DIRS  = {".git", "__pycache__", "node_modules", ".venv", "venv", "dist", "build"}
SKIP_FILES = {"setup.py", "conftest.py"}

def ingest_repo(repo_path: str):
    """
    Walk a repo directory, chunk all Python and JS files,
    embed and store in code_base collection.
    """
    repo_path = Path(repo_path)
    all_chunks = []

    for filepath in repo_path.rglob("*"):
        # Skip hidden dirs and common non-code dirs
        if any(part in SKIP_DIRS for part in filepath.parts):
            continue
        if filepath.name in SKIP_FILES:
            continue
        if filepath.suffix not in CHUNKERS:
            continue

        chunker = CHUNKERS[filepath.suffix]
        chunks  = chunker(filepath)

        if chunks:
            print(f"  {filepath.relative_to(repo_path)} — {len(chunks)} chunks")
            all_chunks.extend(chunks)

    print(f"\nTotal: {len(all_chunks)} code chunks")

    if not all_chunks:
        print("No chunks found — check the path")
        return

    # Embed all at once (batch is much faster than one by one)
    print("Embedding...")
    texts   = [c["text"] for c in all_chunks]
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

    # Upsert to code_base collection
    code_collection.upsert(
        ids        = [c["id"]       for c in all_chunks],
        embeddings = [v.tolist()    for v in vectors],
        documents  = [c["text"]     for c in all_chunks],
        metadatas  = [{
            "source":   c["source"],
            "filename": c["filename"],
            "name":     c["name"],
            "type":     c["type"],
            "line":     c["line"]
        } for c in all_chunks]
    )

    print(f"✓ Upserted {len(all_chunks)} chunks to code_base")

In [0]:
import subprocess

result = subprocess.run(
    ["git", "clone",
     "https://github.com/ariamostajeran/aria-portfolio.git",
     "/tmp/aria-portfolio"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

# Then ingest it
ingest_repo("/tmp/aria-portfolio")
print(f"\ncode_base total: {code_collection.count()} chunks")

In [0]:
def search_code_test(query: str, n: int = 3):
    query_vector = model.encode([query]).tolist()
    results = code_collection.query(
        query_embeddings=query_vector,
        n_results=n,
        include=["documents", "metadatas", "distances"]
    )

    print(f"Query: '{query}'\n")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = round((1 - dist) * 100, 1)
        print(f"{similarity}% — {meta['filename']} / {meta['name']} ({meta['type']})")
        print(f"Line {meta['line']}")
        print(f"Preview: {doc[:200]}")
        print()

search_code_test("how does the data fetcher cache work")
search_code_test("chart rendering javascript")
search_code_test("flask routes")

In [0]:
shutil.make_archive(
    "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/chroma_backup",
    "zip",
    CHROMA_PATH
)
print(f"✓ Backup saved — knowledge: {chroma_client.get_or_create_collection('knowledge_base').count()}, code: {code_collection.count()}")